# Ten inhomogeneous surfaces

The anchors are drawn at random, no two closer than `min_dist`. One master
generator seeds both the anchors and the draw, so the ten surfaces are
reproducible; changing `min_dist` changes the number of rejected candidates
and therefore every surface drawn afterwards. Runtime is about fifteen
minutes.

In [ ]:
import sys, time, pickle
sys.path.insert(0, '..')          # drop this line if the notebook is at the root

import numpy as np
from joblib import Parallel, delayed

from ibo import (sigma_estimator, get_automatic_kernel_args,
                 evaluate_single_bo, kim1,
                 get_benchmarks, shift_benchmarks,
                 sample_gp_function_, generate_centers,
                 plot_grouped_comparison, plot_mean_ratio_across_functions,
                 report_final_gap, paired_comparison)

In [ ]:
n_functions = 10
functions = {}
surface_centers = {}

master = np.random.default_rng(2026)
for i in range(n_functions):
    seed_i = int(master.integers(0, 1_000_000))
    centers = generate_centers(min_dist=0.4, rng=master)
    f_val, _, _ = sample_gp_function_(centers, seed_i)
    functions[f"f_{i+1}"] = f_val
    surface_centers[f"f_{i+1}"] = centers

bounds_ = np.array([[0, 1], [0, 1]])
d = bounds_.shape[0]

tree_depth = 3
n_init     = 40 * d
n_iter     = 200
n_seeds    = 30
min_leaf   = 16
tau        = 1.5          # splitting threshold, shared across all suites
noise_var  = 1e-8

In [ ]:
kernel_args_nstat = get_automatic_kernel_args(
    functions, d=d, n_points=n_init + n_iter, bounds=(0.0, 1.0), nu=2.5)
kernel_args_nstat

In [ ]:
configs = [
    # --- EI Group ---
    ("Classic BO - EI", 'fixed', 1,  False, 'ei', {'xi': 0.0}),
    ("IBO_1 - EI",      'fixed', 1,  True,  'ei', {'xi': 0.0}),
    ("IBO_4 - EI",      'fixed', 2,  True,  'ei', {'xi': 0.0}),
    ("IBO_P - EI",      'tree',  2,  True,  'ei', {'xi': 0.0}),

    # --- PI Group ---
    ("Classic BO - PI", 'fixed', 1,  False, 'pi', {'xi': 0.0}),
    ("IBO_1 - PI",      'fixed', 1,  True,  'pi', {'xi': 0.0}),
    ("IBO_4 - PI",      'fixed', 2,  True,  'pi', {'xi': 0.0}),
    ("IBO_P - PI",      'tree',  2,  True,  'pi', {'xi': 0.0}),

    # --- LCB Group ---
    ("Classic BO - LCB", 'fixed', 1, False, 'lcb', {'beta': 1.96}),
    ("IBO_1 - LCB",      'fixed', 1, True,  'lcb', {'beta': 1.96}),
    ("IBO_4 - LCB",      'fixed', 2, True,  'lcb', {'beta': 1.96}),
    ("IBO_P - LCB",      'tree',  2, True,  'lcb', {'beta': 1.96}),

    # --- TS Group ---
    ("Classic BO - TS", 'fixed', 1, False, 'ts', {}),
    ("IBO_1 - TS",      'fixed', 1, True,  'ts', {}),
    ("IBO_4 - TS",      'fixed', 2, True,  'ts', {}),
    ("IBO_P - TS",      'tree',  2, True,  'ts', {}),
]

In [ ]:
all_histories_new = {}
leaf_report, split_report = {}, {}
start = time.perf_counter()

for f_id, (f_name, objective_function) in enumerate(functions.items()):
    print(f"\nRunning {f_name} ({f_id+1}/{len(functions)}) in parallel...")

    results = Parallel(n_jobs=-1)(
        delayed(evaluate_single_bo)(
            seed, objective_function, bounds_, configs, n_init, d, n_iter,
            kernel_args_nstat[f_name], noise_var, tree_depth, sigma_estimator,
            tau=tau, n_candidates=256, min_leaf=min_leaf, return_info=True,
        ) for seed in range(n_seeds)
    )

    histories_ = {c[0]: np.zeros((n_seeds, n_iter + 1)) for c in configs}
    for seed, seed_histories, infos in results:
        for label, history in seed_histories.items():
            histories_[label][seed, :] = history
    all_histories_new[f_name] = histories_

    # Near 1 means IBO_P has reduced to IBO_1 and tau is too high for
    # genuinely inhomogeneous data.
    leaves = [info["n_leaves_hist"].mean()
              for _, _, infos in results
              for label, info in infos.items() if label.startswith("IBO_P")]
    leaf_report[f_name] = float(np.mean(leaves))
    split_report[f_name] = float(np.mean([l > 1 for l in leaves]))
    print(f"   IBO_P mean leaves {leaf_report[f_name]:.2f}, "
          f"split in {100*split_report[f_name]:.0f}% of runs, "
          f"l = {kernel_args_nstat[f_name]['l']:.2f}")

print(f"\nExecution time: {time.perf_counter() - start:.1f} s")
print(f"IBO_P across the {len(functions)} surfaces (tau = {tau}): "
      f"mean leaves {np.mean(list(leaf_report.values())):.2f}, "
      f"split in {100*np.mean(list(split_report.values())):.1f}% of runs")

In [ ]:
plot_mean_ratio_across_functions(all_histories_new,
                                save_path='f_nstat_10_mean_ratio.pdf')

In [ ]:
report_final_gap(all_histories_new, f_min=1.0)

In [ ]:
nr = {n: {lab: (np.asarray(h, float) - 1.0) / (np.asarray(h, float)[:, [0]] - 1.0)
          for lab, h in hs.items()}
      for n, hs in all_histories_new.items()}
_ = paired_comparison(nr, f_min=0.0)

The paired differences of both suites side by side, one point per function.

In [ ]:
from ibo import plot_paired_differences

with open('../results/bench_tau1.5_leaf16_init80_iter200.pkl', 'rb') as fh:
    bench = pickle.load(fh)['histories']

plot_paired_differences(bench, all_histories_new,
                        save_path='paired_differences.pdf')

In [ ]:
tag = f"tau{tau}_leaf{min_leaf}_init{n_init}_iter{n_iter}"
with open(f'../results/nstat_{tag}.pkl', 'wb') as fh:
    pickle.dump({'histories': all_histories_new,
                 'leaves': leaf_report, 'splits': split_report,
                 'kernel_args': kernel_args_nstat,
                 'centers': surface_centers,
                 'params': {'tau': tau, 'min_leaf': min_leaf, 'n_init': n_init,
                            'n_iter': n_iter, 'n_seeds': n_seeds,
                            'tree_depth': tree_depth}}, fh)
print(f'saved nstat_{tag}.pkl')